In [12]:
import pynmrstar
import csv

def _first(sf, tag):
    try:
        v = sf.get_tag(tag)
        return None if v in ('.','?') else v
    except Exception:
        return None

def _loops_by_category(sf, wanted):
    """Filter loops by their category (e.g., 'Peak_char')."""
    out = []
    for lp in getattr(sf, 'loops', []):
        try:
            cat = lp.category  # works on modern versions
        except Exception:
            # fallback: infer from first tag like '_Peak_char.Chem_shift_val'
            if lp.tags:
                t0 = lp.tags[0]
                cat = t0.split('.')[0].lstrip('_')
            else:
                cat = None
        if cat == wanted:
            out.append(lp)
    return out

def _loop_rows_as_dicts(loop):
    """Return list[dict]; handles STAR placeholders '.'/'?' as None."""
    rows = []
    tags = loop.tags  # full tags like '_Peak_char.Chem_shift_val'
    for r in loop.data:  # list of lists of string values
        d = {}
        for t, v in zip(tags, r):
            d[t] = None if v in ('.','?') else v
        rows.append(d)
    return rows

def extract_peaklist_h1(nmrstar_path, out_csv=None):
    entry = pynmrstar.Entry.from_file(nmrstar_path)

    # ---- compound identifiers (from chem_comp saveframe) ----
    name = inchi = pubchem = None
    for sf in entry.get_saveframes_by_category('chem_comp'):
        name  = _first(sf, 'Chem_comp.Name') or name
        inchi = _first(sf, 'Chem_comp.InChI_code') or inchi
        # optional PubChem
        for lp in _loops_by_category(sf, 'Chem_comp_db_link'):
            for row in _loop_rows_as_dicts(lp):
                db = row.get('_Chem_comp_db_link.Database_code')
                acc_type = (row.get('_Chem_comp_db_link.Accession_code_type') or '').lower()
                if db == 'PubChem' and acc_type in ('cid','sid'):
                    pubchem = row.get('_Chem_comp_db_link.Accession_code') or pubchem

    # ---- sample conditions ----
    pH = temperature_K = None
    for sf in entry.get_saveframes_by_category('sample_conditions'):
        for lp in _loops_by_category(sf, 'Sample_condition_variable'):
            for row in _loop_rows_as_dicts(lp):
                if row.get('_Sample_condition_variable.Type') == 'pH':
                    pH = row.get('_Sample_condition_variable.Val') or pH
                if row.get('_Sample_condition_variable.Type') == 'temperature':
                    temperature_K = row.get('_Sample_condition_variable.Val') or temperature_K

    # ---- spectrometer ----
    field_MHz = None
    for sf in entry.get_saveframes_by_category('NMR_spectrometer'):
        field_MHz = _first(sf, 'NMR_spectrometer.Field_strength') or field_MHz

    # ---- locate 1D 1H spectral peak saveframe ----
    h1_sf = None
    for sf in entry.get_saveframes_by_category('spectral_peak_list'):
        # robust checks: nucleus and/or experiment name
        nucleus = _first(sf, 'Spectral_peak_list.Observed_nucleus')
        expname = _first(sf, 'Spectral_peak_list.Experiment_name') or ''
        if (nucleus and '1H' in nucleus) or expname.startswith('1D 1H'):
            h1_sf = sf
            break
    if h1_sf is None:
        raise RuntimeError("No 1D 1H spectral_peak_list saveframe found")

    # ---- Peak_char: ppm + multiplicity ----
    peak_char = {}
    for lp in _loops_by_category(h1_sf, 'Peak_char'):
        for row in _loop_rows_as_dicts(lp):
            try:
                pid = int(row['_Peak_char.Peak_ID'])
                ppm = float(row['_Peak_char.Chem_shift_val'])
            except Exception:
                continue
            mult = row.get('_Peak_char.Coupling_pattern') or ''
            peak_char[pid] = {'ppm': ppm, 'multiplicity': mult}

    # ---- Peak_general_char: intensity ----
    for lp in _loops_by_category(h1_sf, 'Peak_general_char'):
        for row in _loop_rows_as_dicts(lp):
            try:
                pid = int(row['_Peak_general_char.Peak_ID'])
                inten = float(row['_Peak_general_char.Intensity_val'])
            except Exception:
                continue
            if pid in peak_char:
                peak_char[pid]['intensity'] = inten

    # ---- collect peaks and sort by ppm ----
    peaks = [v for v in peak_char.values() if 'intensity' in v]
    peaks.sort(key=lambda d: d['ppm'])

    # optional write-out
    if out_csv:
        with open(out_csv, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['ppm','intensity','multiplicity'])
            for p in peaks:
                w.writerow([p['ppm'], p['intensity'], p.get('multiplicity','')])

    return {
        'compound_name': name,
        'inchi': inchi,
        'pubchem': pubchem,
        'pH': pH,
        'temperature_K': temperature_K,
        'field_MHz': field_MHz,
        'n_peaks': len(peaks),
        'peaks': peaks,
    }

# Example:
info = extract_peaklist_h1("bmrb_nmrstar/bmse000001/bmse000001.str", out_csv="bmse000001_h1.csv")
print(info['compound_name'], info['inchi'], info['n_peaks'])


AttributeError: 'list' object has no attribute 'startswith'